# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Asif-Ahmed-Rezvi/flyrank-internship-ml/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [21]:
import os

REPO_DIR = "/content/flyrank-internship-ml"
REPO_URL = "https://github.com/Asif-Ahmed-Rezvi/flyrank-internship-ml"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

%cd /content/flyrank-internship-ml

print("Repository ready.")
print("Working directory:", os.getcwd())

/content/flyrank-internship-ml
Repository ready.
Working directory: /content/flyrank-internship-ml


In [22]:
required_files = [
    "skills/README.md",
    "skills/framing-ml-problems/SKILL.md",
    "skills/flyrank/flyrank-data/SKILL.md",
    "docs/ml-intern-dataset-and-lane-guide.md",
    "data/raw/content_refresh_anonymized.csv"
]

for path in required_files:
    print(f"{'✓' if os.path.exists(path) else '✗'} {path}")

✓ skills/README.md
✓ skills/framing-ml-problems/SKILL.md
✓ skills/flyrank/flyrank-data/SKILL.md
✓ docs/ml-intern-dataset-and-lane-guide.md
✓ data/raw/content_refresh_anonymized.csv


## 1. My lane (or freestyle) and why

**Provisional lane: Refresh / Content Opportunity Scoring**

I want to investigate which pages should be reviewed first for a possible refresh or other content action. I chose this lane because content teams have limited review time, so the useful decision is not simply identifying declining pages, but deciding which pages deserve attention first.

The intended output is a transparent ranked review queue that can help an SEO or content editor decide where to focus. I would start with simple, interpretable rules rather than assuming that a machine-learning model is necessary. ML would only be useful if combining multiple observable signals improves the prioritization enough to justify the additional complexity.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(f"Rows/pages: {len(df):,}")
print(f"Clients: {df['client_id'].nunique()}")

Rows/pages: 30,000
Clients: 32


## 2. The question: decision, action, cost of a wrong call

**Search question:** Among pages with observable search and content signals, which pages should an editor review first for a possible refresh or other content action?

**Unit of analysis:** One pseudonymized content item/page.

**Decision:** Which pages deserve limited editorial review time first?

**Output:** A ranked page-level opportunity queue containing a priority score, reason codes, useful context, and a suggested action such as refresh, expand, protect, prune, or monitor.

**Action:** An SEO/content editor can inspect the highest-ranked pages first and decide whether the available evidence supports an intervention.

**Cost of a wrong recommendation:** A false positive could waste editor time or lead to an unnecessary change to a page that was performing acceptably. A false negative could leave a meaningful decline unattended and miss an opportunity to recover visibility or clicks.

**Why data or ML can help:** Prioritization may depend on several signals together, including visibility, position, CTR, content age, freshness, and other content characteristics. I would first compare transparent rules. ML is useful only if it improves the decision-support ranking on honest holdout data enough to justify its additional complexity.

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check the proposed unit of analysis: one content item/page per row

print(f"Rows in dataset: {len(df):,}")
print(f"Unique content IDs: {df['content_id'].nunique():,}")
print(f"Unique clients: {df['client_id'].nunique()}")

if len(df) == df["content_id"].nunique():
    print("Check passed: each row represents one unique content item/page.")
else:
    print("Check needed: some content IDs appear more than once.")

Rows in dataset: 30,000
Unique content IDs: 30,000
Unique clients: 32
Check passed: each row represents one unique content item/page.


## 3. Quick look at the data (2-3 real numbers)

The starter dataset contains **30,000 pages across 32 pseudonymized clients.**

Of these pages, **16,262 (54.2%)** are observed as declining. More importantly for this lane, **9,961 declining pages still receive at least 500 impressions over 90 days**. The median declining page receives **961 impressions over 90 days**.

These observations make the Refresh / Content Opportunity Scoring lane worth investigating. More than half of the pages are observed as declining, and thousands of those pages still have meaningful search visibility. Asking an editor to review every declining page would therefore not be a practical workflow.

A ranking or prioritization system could potentially help an editor focus limited review time on pages where the observable evidence suggests greater opportunity or risk.

These numbers are descriptive observations from the starter dataset. They do **not** show that refreshing a declining page would cause its search performance to improve.

In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
declining = df["trend_direction"].str.lower().eq("down")
visible = df["impressions_90d"] >= 500
declining_visible = declining & visible

print(f"Dataset: {len(df):,} pages across {df['client_id'].nunique()} clients")
print(f"Declining pages: {declining.sum():,} ({declining.mean():.1%})")
print(f"Declining pages with >=500 impressions/90d: {declining_visible.sum():,}")
print(
    f"Median impressions/90d among declining pages: "
    f"{df.loc[declining, 'impressions_90d'].median():,.0f}"
)

Dataset: 30,000 pages across 32 clients
Declining pages: 16,262 (54.2%)
Declining pages with >=500 impressions/90d: 9,961
Median impressions/90d among declining pages: 961


## 4. Careful words: what I can and can't claim

From the starter dataset, I can describe **observed associations and directional patterns** and investigate whether these signals can support a useful page-review ranking.

For example, I can say that many pages in this dataset are observed as declining while still receiving substantial search impressions. I can also evaluate whether observable signals such as impressions, position, CTR, content age, and freshness are useful for prioritizing pages for human review.

I cannot claim that refreshing a page **causes** its search performance to recover, that a high opportunity score guarantees improvement, or that a model has discovered how Google's ranking algorithm works.

The starter data is observational and mainly contains snapshot or trailing-window measurements. Stronger conclusions would require careful feature and target definitions, appropriate temporal data, and honest validation.

I also need to avoid **target leakage**. For example, if a decline label is derived from `trend_direction` or `trend_pct`, those fields should not be used as predictors for that label because they already contain information about the outcome.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.